In [2]:
!pip install jsonschema

In [47]:
import os
import json
import requests
import re
import joblib
import pandas as pd
from jsonschema import validate,ValidationError

In [49]:
from sklearn.model_selection import train_test_split
y_clf=(df["SalePrice"] > df["SalePrice"].median()).astype(int)
X=df.drop(columns=["SalePrice"])
categorical_columns=X.select_dtypes(include=["object","category"]).columns
X=pd.get_dummies(X,columns=categorical_columns,drop_first=True)
X_train,X_test,y_clf_train,y_clf_test=train_test_split(X,y_clf,test_size=0.2,random_state=42,stratify=y_clf)

In [48]:
best_model=joblib.load("best_model.pkl")
print("Model load successfully")
df=pd.read_csv("cleaned_data.csv")
print("Shape:",df.shape)
print(df.head())

Model load successfully
Shape: (2930, 76)
   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Lot Shape Land Contour Utilities  ... Enclosed Porch 3Ssn Porch  \
0       IR1          Lvl    AllPub  ...              0          0   
1       Reg          Lvl    AllPub  ...              0          0   
2       IR1          Lvl    AllPub  ...              0          0   
3       Reg          Lvl    AllPub  ...              0          0   
4       IR1          Lvl    AllPub  ...              0          0   

  Screen Porch Pool Area Misc Val Mo Sold Yr Sold  Sale Type  Sale Condition  \


In [25]:
import getpass
os.environ["LLM_API_KEY"]=getpass.getpass("Enter your openrouter API key:")
print("API key stored sucessfully")

Enter your openrouter API key:··········
API key stored sucessfully


In [33]:
def call_llm(system_prompt,user_prompt,temperature=0.0,max_tokens=512):
  api_key=os.environ["LLM_API_KEY"]
  url="https://openrouter.ai/api/v1/chat/completions"
  headers={
      "Authorization":f"Bearer {api_key}",
      "Content-Type":"application/json"
  }
  payload={
      "model": "openrouter/auto",
      "messages":[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_prompt}
      ],
      "temperature":temperature,
      "max_tokens":max_tokens
  }
  response=requests.post(url,headers=headers,json=payload)
  if response.status_code != 200:
    print("Error:",response.status_code)
    print(response.text)
    return None
  return response.json()["choices"][0]["message"]["content"]

In [59]:
system_prompt="You are a helpful assistant"
user_prompt="Reply with only the word: hello"
result=call_llm(system_prompt,user_prompt)
print(result)

hello


In [60]:
system_prompt="""
you are an AI assistant that explains machine learning predictions.
Return ONLY valid JSON.
The JSON must contain exactly these fields:
{
"prediction_label":"string,
"confidence_level":"low|medium|high",
"top_reason":"string",
"second_reason":"string",
"next_step":"string"
}
Do not include any extra text.
"""

In [37]:
schema={
    "type":"object",
    "properties":{
        "prediction_label":{"type":"string"},
        "confidence_level":{"type":"string"},
        "top_reason":{"type":"string"},
        "second_reason":{"type":"string"},
        "next_step":{"type":"string"}
    },
    "required":[
        "prediction_label","confidence_level","top_reason","second_reason","next_step"
    ]
}

In [39]:
def has_pii(text):
  email_pattern=r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
  phone_pattern=r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
  return bool(
      re.search(email_pattern,text) or
      re.search(phone_pattern,text)
  )

In [41]:
text1="My email is meena@gmail.com"
text2="My house has 1 bedroom and 1 bathroom."
print("Text 1:",has_pii(text1))
print("Text 2:",has_pii(text2))

Text 1: True
Text 2: False


In [42]:
user_input="My email is meena@gmail.com"
if has_pii(user_input):
  print("Input blocked: PII detected.")
else:
  response=call_llm(system_prompt,user_prompt)
  print(response)

Input blocked: PII detected.


In [43]:
user_input="My house has 1 bedroom."
if has_pii(user_input):
  print("Input blocked: PII detected.")
else:
  response=call_llm(system_prompt,user_prompt)
  print(response)

{"prediction_label":"hello","confidence_level":"high","top_reason":"The user explicitly requested hello.","second_reason":"The requested response is unambiguous.","next_step":"No further action."}


In [50]:
sample1= X_test.iloc[[0]]
sample2= X_test.iloc[[1]]
sample3= X_test.iloc[[2]]
print("Three sample inputs created.")

Three sample inputs created.


In [51]:
samples=[sample1,sample2,sample3]
prediction_results=[]
for i,sample in enumerate(samples,start=1):
  prediction=best_model.predict(sample)[0]
  probability=best_model.predict_proba(sample)[0][1]
  prediction_results.append({
      "Sample":i,
      "Prediction":prediction,
      "Probablity":probability
  })
  print(f"Sample {i}")
  print("Prediction:",prediction)
  print("Probablity:",probability)
  print("-"*40)

Sample 1
Prediction: 1
Probablity: 0.7569655791915143
----------------------------------------
Sample 2
Prediction: 1
Probablity: 0.9880131918730529
----------------------------------------
Sample 3
Prediction: 1
Probablity: 0.6592033439703678
----------------------------------------


In [63]:
for i,sample in enumerate(samples,start=1):
  prediction=best_model.predict(sample)[0]
  probability=best_model.predict_proba(sample)[0][1]
  feature_values=sample.to_dict(orient="records")[0]
  user_prompt=f"""
  Feature Values: {json.dumps(feature_values,indent=2)}
  Predicted Class: {prediction}
  Prediction Probablility: {probability:.3f}
  Explain this prediction.
  Return only valid JSON"""
  print(f"Sample{i}")
  print("Feature values:")
  print(feature_values)
  print("Prediction:",prediction)
  print("Probability:",probability)
  if has_pii(user_prompt):
    print("Input blocked: PII detected.")
    print("="*60)
    continue
  response=call_llm(system_prompt,user_prompt)
  print("Raw response:")
  print(response)
  if response is None:
    print("No response recieved.")
    print("="*60)
    continue
  try:
    response=response.strip()
    response=response.replace("```json","")
    response=response.replace("```","")
    explanation=json.loads(response)
    validate(instance=explanation,schema=schema)
    print("Validation status:PASS")
    print(explanation)
  except json.JSONDecodeError:
    print("Validation status:FAIL")
    explanation={
        "prediction_label":None,
        "confidence_level":None,
        "top_reason":None,
        "second_reason":None,
        "next_step":None
    }
    print(explanation)
  except ValidationError:
    print("Validation status:FAIL")
    explanation={
        "prediction_label":None,
        "confidence_level":None,
        "top_reason":None,
        "second_reason":None,
        "next_step":None
    }
    print(explanation)
  print("="*60)

Sample1
Feature values:
{'Order': 2045, 'PID': 904100100, 'MS SubClass': 70, 'Lot Frontage': 107.0, 'Lot Area': 12888, 'Overall Qual': 7, 'Overall Cond': 8, 'Year Built': 1937, 'Year Remod/Add': 1980, 'Mas Vnr Area': 0.0, 'BsmtFin SF 1': 288.0, 'BsmtFin SF 2': 0.0, 'Bsmt Unf SF': 717.0, 'Total Bsmt SF': 1005.0, '1st Flr SF': 1262, '2nd Flr SF': 1005, 'Low Qual Fin SF': 0, 'Gr Liv Area': 2267, 'Bsmt Full Bath': 1.0, 'Bsmt Half Bath': 0.0, 'Full Bath': 1, 'Half Bath': 1, 'Bedroom AbvGr': 3, 'Kitchen AbvGr': 1, 'TotRms AbvGrd': 7, 'Fireplaces': 2, 'Garage Yr Blt': 1937.0, 'Garage Cars': 2.0, 'Garage Area': 498.0, 'Wood Deck SF': 521, 'Open Porch SF': 0, 'Enclosed Porch': 0, '3Ssn Porch': 0, 'Screen Porch': 0, 'Pool Area': 0, 'Misc Val': 0, 'Mo Sold': 4, 'Yr Sold': 2007, 'MS Zoning_C (all)': False, 'MS Zoning_FV': False, 'MS Zoning_I (all)': False, 'MS Zoning_RH': False, 'MS Zoning_RL': True, 'MS Zoning_RM': False, 'Street_Pave': True, 'Lot Shape_IR2': False, 'Lot Shape_IR3': False, 'Lot S

In [58]:
temperature_results=[]
for i,sample in enumerate(samples,start=1):
  prediction=best_model.predict(sample)[0]
  probability=best_model.predict_proba(sample)[0][1]
  feature_values=sample.to_dict(orient="records")[0]
  user_prompt=f"""
  Feature Values: {json.dumps(feature_values,indent=2)}
  Predicted Class: {prediction}
  Prediction Probablility: {probability:.3f}
  Explain this prediction.
  Return only valid JSON"""
  if has_pii(user_prompt):
    print(f"Sample{i}")
    print("Input blocked: PII detected.")
    print("="*60)
    continue
  response_temp0=call_llm(system_prompt,user_prompt,temperature=0)
  response_temp07=call_llm(system_prompt,user_prompt,temperature=0.7)
  print(f"Sample{i}")
  print("-"*60)
  print("Temperature = 0")
  print(response_temp0)
  print()
  print("Temperature = 0.7")
  print(response_temp07)
  print("="*60)
  temperature_results.append({
      "Sample":i,
      "Temperature 0":response_temp0,
      "Temperature 0.7":response_temp07,
  })

Sample1
------------------------------------------------------------
Temperature = 0
```json
{
  "prediction_label": "1",
  "confidence_level": "high",
  "top_reason": "The house has a high overall quality (7) and condition (8), indicating good construction and maintenance.",
  "second_reason": "The property features a substantial living area (2267 sq ft) with a 2-story design, suggesting a larger and potentially more valuable home.",
  "next_step": "Consider analyzing the impact of 'Neighborhood' and 'Year Built' on the sale price, as these are often significant factors in real estate valuation."
}
```

Temperature = 0.7
```json
{
  "prediction_label": "1",
  "confidence_level": "high",
  "top_reason": "The house has a high overall quality (7) and condition (8), indicating good construction and maintenance.",
  "second_reason": "The large 'Gr Liv Area' (2267 sq ft) and 'Total Bsmt SF' (1005 sq ft) suggest a spacious property, which is generally desirable.",
  "next_step": "Consider an